# NL-to-SQL Research Demo
## Comparing Prompting, Fine-tuning, and Agentic Repair on ClassicModels

**What this notebook does:**  
This is a self-contained, visual walkthrough of the dissertation research journey. No model loading or database connection is required — all examples use pre-computed outputs from the real evaluation runs.

---

### Research Question
> *Can few-shot prompting alone outperform QLoRA fine-tuning for NL-to-SQL on a closed-domain database, in a resource-constrained setting?*

### The Three Approaches Evaluated

| Approach | Description | When it matters |
|---|---|---|
| **Baseline (zero/few-shot)** | Prompt the model with 0 or 3 examples | Fast, no training cost |
| **QLoRA fine-tuning** | 4-bit quantised LoRA adapters trained on domain data | Can a tuned model beat prompting? |
| **ReAct loop** | Generate → validate → repair using execution feedback | Can iterative self-correction add value? |

### Models
- **Llama-3-8B-Instruct** (Meta)  
- **Qwen-2.5-7B-Instruct** (Alibaba)  

Both run in 4-bit NF4 quantisation on a single Colab T4 GPU (~16 GB VRAM).

In [ ]:
# Standard imports — this demo runs without a GPU, model, or database
import textwrap
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import numpy as np
from IPython.display import display, Markdown, HTML

plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'monospace',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Imports OK — demo is self-contained, no GPU required.')

---
## 1. The Database: ClassicModels

All experiments run against a single fixed MySQL database — **ClassicModels** — a standard retail dataset with 8 tables covering customers, orders, products, and payments.  
The benchmark contains **200 hand-crafted NL questions** paired with gold SQL answers.

In [ ]:
# ── Schema entity-relationship diagram ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(0, 13)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_facecolor('#f8f9fa')
fig.patch.set_facecolor('#f8f9fa')

# Table definitions: (name, x, y, columns)
tables = [
    ('customers',    1.0,  4.8, ['customerNumber (PK)', 'customerName', 'country', 'creditLimit', '...']),
    ('orders',       4.5,  4.8, ['orderNumber (PK)', 'customerNumber (FK)', 'status', 'orderDate', '...']),
    ('orderdetails', 8.0,  4.8, ['orderNumber (FK)', 'productCode (FK)', 'quantityOrdered', 'priceEach']),
    ('products',     11.0, 4.8, ['productCode (PK)', 'productName', 'productLine (FK)', 'MSRP', '...']),
    ('productlines', 11.0, 1.8, ['productLine (PK)', 'textDescription']),
    ('payments',     1.0,  1.8, ['customerNumber (FK)', 'checkNumber', 'amount', 'paymentDate']),
    ('employees',    4.5,  1.8, ['employeeNumber (PK)', 'reportsTo (FK)', 'jobTitle', '...']),
    ('offices',      8.0,  1.8, ['officeCode (PK)', 'city', 'country', 'phone']),
]

box_w, box_h = 2.0, 1.85
header_color = '#2c3e50'
box_color    = '#ecf0f1'
text_color   = '#2c3e50'

table_centers = {}
for name, x, y, cols in tables:
    cx = x + box_w / 2
    cy = y
    table_centers[name] = (cx, cy)

    rect = mpatches.FancyBboxPatch(
        (x, y - box_h + 0.3), box_w, box_h,
        boxstyle='round,pad=0.05', linewidth=1.5,
        edgecolor=header_color, facecolor=box_color
    )
    ax.add_patch(rect)

    # Header bar
    header = mpatches.FancyBboxPatch(
        (x, y - 0.02), box_w, 0.35,
        boxstyle='round,pad=0.02', linewidth=0,
        facecolor=header_color
    )
    ax.add_patch(header)
    ax.text(cx, y + 0.15, name, ha='center', va='center',
            fontsize=7.5, fontweight='bold', color='white')

    for i, col in enumerate(cols):
        ax.text(x + 0.1, y - 0.22 - i * 0.28, col,
                ha='left', va='center', fontsize=5.8, color=text_color)

# FK relationships (from_table, to_table, label)
fk_links = [
    ('orders',       'customers',    'customerNumber'),
    ('orderdetails', 'orders',       'orderNumber'),
    ('orderdetails', 'products',     'productCode'),
    ('products',     'productlines', 'productLine'),
    ('payments',     'customers',    'customerNumber'),
    ('employees',    'offices',      'officeCode'),
]

for src, dst, label in fk_links:
    x1, y1 = table_centers[src]
    x2, y2 = table_centers[dst]
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#7f8c8d',
                                lw=1.2, connectionstyle='arc3,rad=0.05'))
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    ax.text(mx, my + 0.12, label, ha='center', va='bottom',
            fontsize=5.2, color='#7f8c8d')

ax.set_title('ClassicModels Schema  (8 tables, 200-item benchmark)',
             fontsize=11, fontweight='bold', color=text_color, pad=10)
plt.tight_layout()
plt.show()

---
## 2. Evaluation Metrics

Four metrics are used, each measuring a different level of correctness:

| Metric | Full name | What it tests | Score range |
|--------|-----------|---------------|-------------|
| **VA** | Valid SQL | Does the SQL parse and execute without error? | 0–1 |
| **EM** | Exact Match | Does the predicted SQL match the gold string after normalisation? | 0–1 |
| **EX** | Execution Accuracy | Do the predicted and gold queries return the same result set? | 0–1 |
| **TS** | Test-Suite Accuracy | Does EX hold across 10 database replicas with different values? | 0–1 |

> **Why EX > EM?**  Multiple valid SQL queries can answer the same question (`COUNT(*)` vs `COUNT(col)`). EX captures this; EM does not.

In [ ]:
# ── Visual explanation of metrics using one example ─────────────────────────
NLQ  = 'How many customers are in France?'
GOLD = "SELECT COUNT(*) FROM customers WHERE country = 'France';"

# Three hypothetical model outputs
preds = [
    ("SELECT COUNT(*) FROM customers WHERE country = 'France';",
     True,  True,  True,  True,  'Perfect match'),
    ("SELECT COUNT(customerNumber) FROM customers WHERE country = 'France';",
     True,  False, True,  True,  'Correct but different syntax'),
    ("SELECT * FROM customers WHERE country = 'France';",
     True,  False, False, False, 'Wrong projection (SELECT *)'),
    ("SELECT COUNT(*) FROM orders WHERE country = 'France';",
     False, False, False, False, 'Schema error (orders has no country)'),
]

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.axis('off')
fig.patch.set_facecolor('#f8f9fa')

colors = {'✓': '#27ae60', '✗': '#e74c3c'}

# Header
ax.text(0.01, 0.97, f'NLQ:  "{NLQ}"', transform=ax.transAxes,
        fontsize=9, fontweight='bold', va='top', color='#2c3e50')
ax.text(0.01, 0.88, f'Gold: {GOLD}', transform=ax.transAxes,
        fontsize=8, va='top', color='#555', family='monospace')

headers = ['Predicted SQL', 'VA', 'EM', 'EX', 'TS', 'Notes']
col_x   = [0.01, 0.52, 0.59, 0.66, 0.73, 0.80]
row_y_start = 0.74
row_h = 0.155

for j, (hdr, cx) in enumerate(zip(headers, col_x)):
    ax.text(cx, row_y_start, hdr, transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='top', color='#2c3e50')

ax.axhline(y=0.72, color='#aaa', lw=0.8, transform=ax.transAxes)

for i, (sql, va, em, ex, ts, note) in enumerate(preds):
    y = row_y_start - (i + 1) * row_h
    bg = '#ffffff' if i % 2 == 0 else '#f0f0f0'
    ax.add_patch(mpatches.FancyBboxPatch(
        (0, y - 0.04), 1, row_h - 0.01,
        boxstyle='square,pad=0', transform=ax.transAxes,
        facecolor=bg, edgecolor='none', zorder=0
    ))
    wrapped = textwrap.shorten(sql, width=70, placeholder='...')
    ax.text(col_x[0], y, wrapped, transform=ax.transAxes,
            fontsize=6.5, va='top', family='monospace', color='#2c3e50')
    for j, val in enumerate([va, em, ex, ts]):
        sym = '✓' if val else '✗'
        ax.text(col_x[j + 1] + 0.02, y, sym, transform=ax.transAxes,
                fontsize=9, va='top', color=colors[sym], fontweight='bold')
    ax.text(col_x[5], y, note, transform=ax.transAxes,
            fontsize=6.5, va='top', color='#555', style='italic')

ax.set_title('Metric comparison for a single benchmark item',
             fontsize=10, fontweight='bold', color='#2c3e50', pad=4)
plt.tight_layout()
plt.show()

---
## 3. Approach 1 — Baseline Prompting (Zero-shot vs. Few-shot)

The **baseline** passes a system prompt + schema + (optionally) worked examples directly into the model.  
No training, no fine-tuning — just carefully structured text.

### k=0 (zero-shot) vs. k=3 (3-shot)
- **k=0**: The model sees the schema and the question only
- **k=3**: The model also sees 3 NLQ→SQL examples drawn from the benchmark pool

In [ ]:
# ── Visual: prompt construction for k=0 and k=3 ────────────────────────────
SCHEMA_SNIPPET = """customers(customerNumber, customerName, country, creditLimit)
orders(orderNumber, customerNumber, status, orderDate)
orderdetails(orderNumber, productCode, quantityOrdered, priceEach)
products(productCode, productName, productLine, MSRP)
payments(customerNumber, checkNumber, amount)
Join hints: customers.customerNumber = orders.customerNumber; ..."""

EXEMPLARS = [
    ('List all customer names in Spain.',
     "SELECT customerName FROM customers WHERE country = 'Spain';"),
    ('How many products are in each product line?',
     'SELECT productLine, COUNT(*) FROM products GROUP BY productLine;'),
    ('What is the total payment amount received?',
     'SELECT SUM(amount) FROM payments;'),
]

TARGET_NLQ = 'Top 3 countries by number of orders.'

fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), facecolor='#f8f9fa')

def draw_prompt_box(ax, title, sections):
    """sections: list of (label, text, color)"""
    ax.axis('off')
    ax.set_facecolor('#f8f9fa')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_title(title, fontsize=10, fontweight='bold', color='#2c3e50', pad=6)

    y = 9.7
    for label, text, color in sections:
        # label bar
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.1, y - 0.3), 9.8, 0.32,
            boxstyle='round,pad=0.05', facecolor=color, edgecolor='none'
        ))
        ax.text(5, y - 0.14, label, ha='center', va='center',
                fontsize=7.5, fontweight='bold', color='white')
        y -= 0.35
        lines = text.split('\n')
        for line in lines:
            wrapped = textwrap.wrap(line, width=58) or ['']
            for wl in wrapped:
                ax.text(0.3, y, wl, va='top', fontsize=6.2,
                        family='monospace', color='#2c3e50')
                y -= 0.30
        y -= 0.15  # gap between sections

k0_sections = [
    ('[SYSTEM] You are a MySQL analyst. Return one SQL SELECT.',
     '', '#2c3e50'),
    ('[SCHEMA]', SCHEMA_SNIPPET, '#2980b9'),
    ('[USER] Convert to SQL:', TARGET_NLQ, '#27ae60'),
]

exemplar_text = ''
for nlq, sql in EXEMPLARS:
    exemplar_text += f'Q: {nlq}\nA: {sql}\n\n'

k3_sections = [
    ('[SYSTEM] You are a MySQL analyst. Return one SQL SELECT.',
     '', '#2c3e50'),
    ('[SCHEMA]', SCHEMA_SNIPPET, '#2980b9'),
    ('[EXEMPLARS] 3 worked examples:', exemplar_text.strip(), '#8e44ad'),
    ('[USER] Convert to SQL:', TARGET_NLQ, '#27ae60'),
]

draw_prompt_box(axes[0], 'k=0  Zero-shot prompt', k0_sections)
draw_prompt_box(axes[1], 'k=3  Few-shot prompt  (+3 examples)', k3_sections)

plt.tight_layout()
plt.show()
print('k=0: model must generalise from schema alone')
print('k=3: model sees the output format + SQL patterns before answering')

---
## 4. Approach 2 — QLoRA Fine-tuning

**QLoRA** (Dettmers et al., 2023) fine-tunes a quantised model by adding small trainable rank-decomposition matrices (LoRA adapters) to the attention layers. The base model weights stay frozen in 4-bit NF4 format.

- **Training data**: 160 items from the 200-item benchmark (held-out 40 items not used)
- **Hardware**: single T4 GPU (~16 GB VRAM), ~25 minutes per run  
- **Key question**: does domain-specific fine-tuning outperform few-shot prompting with zero training?

In [ ]:
# ── QLoRA architecture diagram ──────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5.5), facecolor='#f8f9fa')
ax.set_xlim(0, 11)
ax.set_ylim(0, 5.5)
ax.axis('off')
ax.set_facecolor('#f8f9fa')

def box(ax, x, y, w, h, label, sublabel='', fc='#2c3e50', tc='white', fs=8):
    r = mpatches.FancyBboxPatch((x, y), w, h,
        boxstyle='round,pad=0.1', facecolor=fc, edgecolor='#aaa', linewidth=1)
    ax.add_patch(r)
    ax.text(x + w/2, y + h/2 + (0.12 if sublabel else 0), label,
            ha='center', va='center', fontsize=fs, fontweight='bold', color=tc)
    if sublabel:
        ax.text(x + w/2, y + h/2 - 0.22, sublabel,
                ha='center', va='center', fontsize=6, color=tc, style='italic')

def arrow(ax, x1, y1, x2, y2, label='', color='#7f8c8d'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    if label:
        ax.text((x1+x2)/2+0.05, (y1+y2)/2, label,
                ha='left', va='center', fontsize=6.5, color=color)

# Frozen base model block
box(ax, 0.3, 1.2, 3.2, 3.0, 'Frozen Base Model', '4-bit NF4 quantised\n~4 GB VRAM', fc='#7f8c8d', fs=9)
ax.text(1.9, 0.85, '🔒 weights frozen', ha='center', fontsize=7, color='#7f8c8d')

# LoRA adapter block (small, overlaid)
box(ax, 3.8, 1.8, 2.4, 1.8, 'LoRA Adapters', 'rank=8, α=16\n~6 M trainable params', fc='#e67e22', fs=8.5)
ax.text(5.0, 1.5, '✏️  only these train', ha='center', fontsize=7, color='#e67e22')

# Merge arrow
arrow(ax, 3.5, 2.7, 3.8, 2.7, color='#e67e22')
ax.text(3.62, 2.9, 'inject', ha='center', fontsize=6.5, color='#e67e22')

# Training data
box(ax, 0.3, 4.6, 2.5, 0.6, 'Training data', '160 NLQ→SQL pairs', fc='#2980b9', fs=8)
arrow(ax, 1.55, 4.6, 1.55, 4.2, color='#2980b9')

# Output
box(ax, 6.6, 2.0, 2.4, 1.6, 'QLoRA Model', 'Base + adapters\nloaded at inference', fc='#27ae60', fs=8.5)
arrow(ax, 6.2, 2.7, 6.6, 2.7, color='#27ae60')

# Input/output labels
box(ax, 9.3, 2.0, 1.5, 0.7, 'SQL output', '', fc='#2c3e50', fs=7)
arrow(ax, 9.0, 2.4, 9.3, 2.4)

box(ax, 9.3, 3.0, 1.5, 0.7, 'NLQ input', '', fc='#2c3e50', fs=7)
arrow(ax, 9.3, 3.35, 9.0, 3.35)

ax.text(5.5, 5.2, 'QLoRA Architecture: small adapters trained on frozen quantised backbone',
        ha='center', fontsize=10, fontweight='bold', color='#2c3e50')

plt.tight_layout()
plt.show()

---
## 5. Approach 3 — ReAct Loop (Generate → Validate → Repair)

The **ReAct** (Yao et al., 2023) pattern interleaves *reasoning* and *acting* in a loop.  
For NL-to-SQL this means:

1. **Get schema** — load table/column definitions  
2. **Generate SQL** — few-shot prompted generation  
3. **Validate SQL** — static schema checks (unknown columns, SELECT \*, ambiguity)  
4. **Validate constraints** — NLQ-derived semantic checks (required columns, aggregations, ORDER BY)  
5. **Execute SQL** — actually run against the database  
6. **Repair** — if any check fails, feed the error back to the model and regenerate  

Steps 3–6 repeat up to `max_repairs` times.

In [ ]:
# ── ReAct loop flowchart ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 9), facecolor='#f8f9fa')
ax.set_xlim(0, 10)
ax.set_ylim(0, 9)
ax.axis('off')

NODE_W, NODE_H = 3.2, 0.6
CX = 5.0  # centre x

def rect_node(ax, cx, cy, w, h, text, fc='#2c3e50', tc='white', fs=8.5):
    r = mpatches.FancyBboxPatch((cx - w/2, cy - h/2), w, h,
        boxstyle='round,pad=0.08', facecolor=fc, edgecolor='#555', lw=1.2)
    ax.add_patch(r)
    ax.text(cx, cy, text, ha='center', va='center',
            fontsize=fs, fontweight='bold', color=tc)
    return (cx - w/2, cy, cx + w/2, cy)  # left_mid, right_mid

def diamond(ax, cx, cy, w, h, text, fc='#f39c12', fs=7.5):
    pts = np.array([
        [cx,     cy + h/2],
        [cx + w/2, cy],
        [cx,     cy - h/2],
        [cx - w/2, cy],
    ])
    poly = plt.Polygon(pts, closed=True, facecolor=fc, edgecolor='#555', lw=1.2)
    ax.add_patch(poly)
    ax.text(cx, cy, text, ha='center', va='center', fontsize=fs, fontweight='bold')

def vert_arrow(ax, x, y1, y2, color='#555', label='', label_side='right'):
    ax.annotate('', xy=(x, y2), xytext=(x, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.4))
    if label:
        lx = x + 0.15 if label_side == 'right' else x - 0.15
        ha = 'left' if label_side == 'right' else 'right'
        ax.text(lx, (y1 + y2)/2, label, ha=ha, va='center', fontsize=7, color=color)

# Nodes top to bottom
rect_node(ax, CX, 8.5, 3.8, 0.6, '① get_schema', fc='#2980b9')
vert_arrow(ax, CX, 8.2, 7.7)
rect_node(ax, CX, 7.5, 3.8, 0.6, '② generate_sql  (k=3 few-shot)', fc='#8e44ad')
vert_arrow(ax, CX, 7.2, 6.7)

diamond(ax, CX, 6.4, 3.6, 0.7, '③ validate_sql')
vert_arrow(ax, CX, 6.05, 5.55, label='valid', label_side='right')
# fail → repair
ax.annotate('', xy=(8.5, 7.5), xytext=(6.8, 6.4),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.4,
                            connectionstyle='arc3,rad=-0.4'))
ax.text(8.0, 7.0, 'invalid\n→ repair', ha='center', va='center',
        fontsize=7, color='#e74c3c')

diamond(ax, CX, 5.2, 3.8, 0.7, '④ validate_constraints')
vert_arrow(ax, CX, 4.85, 4.35, label='valid', label_side='right')
ax.annotate('', xy=(8.5, 7.5), xytext=(6.9, 5.2),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.4,
                            connectionstyle='arc3,rad=-0.5'))
ax.text(8.7, 6.35, 'invalid\n→ repair', ha='center', va='center',
        fontsize=7, color='#e74c3c')

diamond(ax, CX, 4.0, 3.6, 0.7, '⑤ run_sql (execute)')
vert_arrow(ax, CX, 3.65, 3.0, label='success', label_side='right')
ax.annotate('', xy=(8.5, 7.5), xytext=(6.8, 4.0),
            arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1.4,
                            connectionstyle='arc3,rad=-0.55'))
ax.text(8.95, 5.75, 'runtime\nerror\n→ repair', ha='center', va='center',
        fontsize=7, color='#e74c3c')

rect_node(ax, CX, 2.7, 3.8, 0.6, '⑥ STOP — return final SQL', fc='#27ae60')

# Repair node on the right
rect_node(ax, 8.5, 7.5, 2.4, 0.6, 'repair_sql\n(error → hint → LLM)', fc='#e67e22', fs=7.5)

# Budget exhausted
ax.text(CX, 2.1, 'Budget exhausted → return best SQL seen',
        ha='center', va='center', fontsize=7.5, color='#e74c3c', style='italic')

ax.set_title('ReAct Loop: Reason → Act → Observe → Repair',
             fontsize=11, fontweight='bold', color='#2c3e50', pad=10)
plt.tight_layout()
plt.show()

---
## 6. ReAct Loop — Live Trace Walkthrough

The two traces below come directly from the real evaluation run (`results/agent/results_react_200.json`).  
Both show a case where the first SQL generated was **invalid** and the loop successfully **repaired** it.

In [ ]:
# ── Example 1: missing JOIN causes unknown_column error ─────────────────────
# (real trace from evaluation run, item: 'Top 3 countries by number of orders')

trace_1 = [
    {'step': 1, 'action': 'get_schema',
     'obs': 'Loaded 8 tables: customers, orders, orderdetails, products, …'},
    {'step': 2, 'action': 'generate_sql',
     'obs': "SELECT country, COUNT(orderNumber) AS order_count\n"
            "FROM orders\nGROUP BY country\nORDER BY order_count DESC LIMIT 3;",
     'tag': 'generated'},
    {'step': 3, 'action': 'validate_sql',
     'obs': 'FAIL — unknown_column: orders.country (country is in customers, not orders)',
     'tag': 'fail'},
    {'step': 4, 'action': 'repair_sql',
     'obs': "SELECT c.country, COUNT(o.orderNumber) AS order_count\n"
            "FROM customers c\nJOIN orders o ON c.customerNumber = o.customerNumber\n"
            "GROUP BY c.country ORDER BY order_count DESC LIMIT 3;",
     'tag': 'repaired',
     'hint': 'Repair hint: unknown_column: country. Schema shows customers.country.'},
    {'step': 5, 'action': 'validate_sql',    'obs': 'OK — schema_ok',           'tag': 'pass'},
    {'step': 6, 'action': 'validate_constraints', 'obs': 'OK',                  'tag': 'pass'},
    {'step': 7, 'action': 'run_sql',         'obs': 'SUCCESS — 3 rows returned', 'tag': 'pass'},
    {'step': 7, 'action': 'STOP',            'obs': 'reason: success',           'tag': 'stop'},
]

NLQ_1   = 'Top 3 countries by number of orders.'
GOLD_1  = ('SELECT c.country, COUNT(*) AS order_count\n'
           'FROM customers c JOIN orders o ON c.customerNumber = o.customerNumber\n'
           'GROUP BY c.country ORDER BY order_count DESC LIMIT 3;')

TAG_COLORS = {
    'generated': '#8e44ad',
    'fail':      '#e74c3c',
    'repaired':  '#e67e22',
    'pass':      '#27ae60',
    'stop':      '#2980b9',
    None:        '#2c3e50',
}

def render_trace(trace, nlq, gold, title):
    fig, ax = plt.subplots(figsize=(13, len(trace) * 0.88 + 1.2), facecolor='#f8f9fa')
    ax.axis('off')
    ax.set_facecolor('#f8f9fa')
    total_h = len(trace) * 0.88 + 1.2
    ax.set_xlim(0, 13)
    ax.set_ylim(0, total_h)

    # Header
    ax.text(0.2, total_h - 0.15, f'NLQ:  "{nlq}"',
            va='top', fontsize=9, fontweight='bold', color='#2c3e50')
    ax.text(0.2, total_h - 0.52, f'Gold: {gold.split(chr(10))[0]}…',
            va='top', fontsize=7.5, color='#555', family='monospace')

    y = total_h - 1.1
    for t in trace:
        color = TAG_COLORS.get(t.get('tag'), '#2c3e50')
        # step badge
        ax.add_patch(mpatches.FancyBboxPatch(
            (0.1, y - 0.32), 0.55, 0.44,
            boxstyle='round,pad=0.04', facecolor=color, edgecolor='none'
        ))
        ax.text(0.38, y - 0.1, f"#{t['step']}", ha='center', va='center',
                fontsize=7.5, fontweight='bold', color='white')

        # action label
        ax.text(0.75, y, t['action'], va='top', fontsize=8,
                fontweight='bold', color=color)

        # observation (possibly multi-line SQL)
        obs_lines = t['obs'].split('\n')
        for li, line in enumerate(obs_lines):
            ax.text(2.7, y - li * 0.26,
                    textwrap.shorten(line, width=95, placeholder='…'),
                    va='top', fontsize=6.8, family='monospace', color='#2c3e50')

        # repair hint if present
        if t.get('hint'):
            ax.text(2.7, y - len(obs_lines) * 0.26,
                    f"↳ {t['hint']}",
                    va='top', fontsize=6.2, color='#e67e22', style='italic')

        # connector line
        ax.plot([0.38, 0.38], [y - 0.32, y - 0.75], color='#ccc', lw=1)

        y -= 0.88

    ax.set_title(title, fontsize=10, fontweight='bold', color='#2c3e50', pad=6)
    plt.tight_layout()
    plt.show()

render_trace(trace_1, NLQ_1, GOLD_1,
             'Trace 1 — Missing JOIN detected and repaired automatically')

In [ ]:
# ── Example 2: ambiguous column name repaired by adding table alias ──────────
# (real trace from evaluation run)

trace_2 = [
    {'step': 1, 'action': 'get_schema',
     'obs': 'Loaded 8 tables: customers, orders, orderdetails, products, …'},
    {'step': 2, 'action': 'generate_sql',
     'obs': "SELECT COUNT(*) AS disputedOrders\n"
            "FROM orders WHERE status = 'Disputed'\n"
            "AND customerNumber IN (SELECT customerNumber FROM customers WHERE country = 'Australia');",
     'tag': 'generated'},
    {'step': 3, 'action': 'validate_sql',
     'obs': 'FAIL — ambiguous_column: customerNumber (exists in both orders and customers)',
     'tag': 'fail'},
    {'step': 4, 'action': 'repair_sql',
     'obs': "SELECT COUNT(*) AS disputedOrders\n"
            "FROM orders o\n"
            "JOIN customers c ON o.customerNumber = c.customerNumber\n"
            "WHERE o.status = 'Disputed' AND c.country = 'Australia';",
     'tag': 'repaired',
     'hint': 'Repair hint: qualify the ambiguous column customerNumber with the correct table alias.'},
    {'step': 5, 'action': 'validate_sql',         'obs': 'OK — schema_ok', 'tag': 'pass'},
    {'step': 6, 'action': 'validate_constraints',  'obs': 'OK',            'tag': 'pass'},
    {'step': 7, 'action': 'run_sql',               'obs': 'SUCCESS — 1 row returned', 'tag': 'pass'},
    {'step': 7, 'action': 'STOP',                  'obs': 'reason: success', 'tag': 'stop'},
]

NLQ_2  = "How many orders from customers in Australia have status 'Disputed'?"
GOLD_2 = ("SELECT COUNT(*) AS order_count\n"
          "FROM orders o JOIN customers c ON o.customerNumber = c.customerNumber\n"
          "WHERE c.country = 'Australia' AND o.status = 'Disputed';")

render_trace(trace_2, NLQ_2, GOLD_2,
             'Trace 2 — Ambiguous column qualified with table alias after repair')

---
## 7. Results — All Conditions Compared

The table below summarises results from the full evaluation across 8 conditions.  
Numbers are **means across seeds** (3 seeds × 200 items = 600 pairs per condition, except Qwen Base k=0 which has 2 seeds).

In [ ]:
# ── Results bar chart (pre-computed means from evaluation CSVs) ──────────────
conditions = [
    'Llama\nBase k=0', 'Llama\nBase k=3', 'Llama\nQLoRA k=0', 'Llama\nQLoRA k=3',
    'Qwen\nBase k=0',  'Qwen\nBase k=3',  'Qwen\nQLoRA k=0',  'Qwen\nQLoRA k=3',
]

# From stats_mean_median_shapiro.csv (mean across seeds)
va = [0.765, 0.870, 0.815, 0.855,  0.805, 0.892, 0.805, 0.895]
em = [0.005, 0.298, 0.000, 0.285,  0.010, 0.357, 0.010, 0.302]
ex = [0.410, 0.517, 0.440, 0.465,  0.480, 0.610, 0.435, 0.530]

x = np.arange(len(conditions))
w = 0.22

fig, ax = plt.subplots(figsize=(14, 5.5), facecolor='#f8f9fa')
ax.set_facecolor('#f8f9fa')

bars_va = ax.bar(x - w,     va, w, label='VA (Valid SQL)',         color='#2980b9', alpha=0.85)
bars_em = ax.bar(x,         em, w, label='EM (Exact Match)',        color='#e74c3c', alpha=0.85)
bars_ex = ax.bar(x + w,     ex, w, label='EX (Execution Accuracy)', color='#27ae60', alpha=0.85)

# Annotate EX bars with value
for bar in bars_ex:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.012,
            f'{h:.2f}', ha='center', va='bottom', fontsize=6.5, color='#27ae60')

# Highlight k=3 conditions with a shaded band
for xi in [1, 3, 5, 7]:
    ax.axvspan(xi - 0.45, xi + 0.45, alpha=0.06, color='#27ae60', zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(conditions, fontsize=8.5)
ax.set_ylabel('Score (0–1)', fontsize=9)
ax.set_ylim(0, 1.05)
ax.legend(loc='upper left', fontsize=8.5)
ax.set_title('Evaluation Results — All 8 Conditions  (green bands = k=3)',
             fontsize=11, fontweight='bold', color='#2c3e50')

# Divider between Llama and Qwen
ax.axvline(3.5, color='#aaa', lw=1, ls='--')
ax.text(1.5, 1.02, '← Llama-3-8B →', ha='center', fontsize=8, color='#7f8c8d')
ax.text(5.5, 1.02, '← Qwen-2.5-7B →', ha='center', fontsize=8, color='#7f8c8d')

plt.tight_layout()
plt.show()

In [ ]:
# ── Key deltas: Base vs QLoRA at k=3 (EX metric, the primary metric) ─────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), facecolor='#f8f9fa')

def delta_chart(ax, model_name, base_k0, base_k3, qlora_k0, qlora_k3):
    labels = ['Base k=0', 'Base k=3', 'QLoRA k=0', 'QLoRA k=3']
    vals   = [base_k0,    base_k3,    qlora_k0,     qlora_k3]
    colors = ['#2980b9', '#1a5276', '#e67e22', '#784212']

    bars = ax.bar(labels, vals, color=colors, width=0.55, alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Delta annotation Base k3 vs QLoRA k3
    delta = base_k3 - qlora_k3
    ax.annotate('', xy=(2.5 + 0.275, base_k3), xytext=(2.5 + 0.275, qlora_k3),
                arrowprops=dict(arrowstyle='<->', color='#e74c3c', lw=2))
    ax.text(2.5 + 0.55, (base_k3 + qlora_k3)/2,
            f'+{delta*100:.1f}pp\n(Base wins)',
            ha='left', va='center', fontsize=8, color='#e74c3c', fontweight='bold')

    ax.set_ylim(0, 0.80)
    ax.set_ylabel('EX (Execution Accuracy)', fontsize=8.5)
    ax.set_title(f'{model_name} — EX by condition', fontsize=10, fontweight='bold', color='#2c3e50')
    ax.set_facecolor('#f8f9fa')
    ax.tick_params(labelsize=8)

delta_chart(axes[0], 'Llama-3-8B', 0.410, 0.517, 0.440, 0.465)
delta_chart(axes[1], 'Qwen-2.5-7B', 0.480, 0.610, 0.435, 0.530)

fig.suptitle('Base outperforms QLoRA at k=3 on both models — the core dissertation finding',
             fontsize=11, fontweight='bold', color='#2c3e50', y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Statistical Significance — Paired t-tests

All differences are tested with **paired t-tests** matched on `(seed, example_id)`.  
Because each metric is a binary 0/1 variable, the distributions are non-normal (Shapiro-Wilk rejects in every case) — but at n=400–600 pairs, the Central Limit Theorem justifies the t-test.

H₀: *mean(condition_A) = mean(condition_B)*  
Significance threshold: α = 0.05

In [ ]:
# ── Statistical significance summary (from stats_paired_ttests.csv) ───────────
results_stats = [
    # (comparison, metric, left_mean, right_mean, p_value, decision)
    ('Llama Base k0→k3',      'EX', 0.410, 0.517, 1.10e-11, 'reject_H0'),
    ('Llama Base k0→k3',      'VA', 0.765, 0.870, 1.81e-11, 'reject_H0'),
    ('Llama QLoRA k0→k3',     'EX', 0.440, 0.465, 0.0547,   'FAIL to reject'),   # key finding
    ('Llama QLoRA k0→k3',     'VA', 0.815, 0.855, 0.0113,   'reject_H0'),
    ('Qwen Base k0→k3',       'EX', 0.480, 0.613, 1.50e-07, 'reject_H0'),
    ('Qwen Base k0→k3',       'VA', 0.805, 0.890, 2.84e-06, 'reject_H0'),
    ('Qwen QLoRA k0→k3',      'EX', 0.435, 0.530, 1.37e-09, 'reject_H0'),
    ('Llama Base→QLoRA @k3',  'EX', 0.517, 0.465, 7.40e-04, 'reject_H0  ★'),
    ('Qwen Base→QLoRA @k3',   'EX', 0.610, 0.530, 8.60e-06, 'reject_H0  ★'),
    ('Qwen Base→QLoRA @k3',   'TS', 0.675, 0.578, 9.29e-07, 'reject_H0  ★'),
]

fig, ax = plt.subplots(figsize=(13, 5.5), facecolor='#f8f9fa')
ax.axis('off')
ax.set_facecolor('#f8f9fa')

col_headers = ['Comparison', 'Metric', 'Left mean', 'Right mean', 'Δ (pp)', 'p-value', 'Decision']
col_x = [0.0, 0.32, 0.45, 0.56, 0.67, 0.76, 0.88]

# Header row
ax.add_patch(mpatches.FancyBboxPatch(
    (-0.01, 0.91), 1.02, 0.07, transform=ax.transAxes,
    boxstyle='square,pad=0', facecolor='#2c3e50', edgecolor='none'
))
for hdr, cx in zip(col_headers, col_x):
    ax.text(cx + 0.005, 0.945, hdr, transform=ax.transAxes,
            fontsize=8, fontweight='bold', va='center', color='white')

for i, (comp, metric, lm, rm, pval, decision) in enumerate(results_stats):
    y = 0.88 - i * 0.084
    bg = '#fff' if i % 2 == 0 else '#f5f5f5'
    highlight = '#fff8e1' if '★' in decision else bg
    fail_row  = 'FAIL' in decision
    highlight = '#fdecea' if fail_row else highlight

    ax.add_patch(mpatches.FancyBboxPatch(
        (-0.01, y - 0.04), 1.02, 0.082, transform=ax.transAxes,
        boxstyle='square,pad=0', facecolor=highlight, edgecolor='none'
    ))

    delta_pp = (rm - lm) * 100
    delta_str = f'{delta_pp:+.1f}'
    p_str = f'{pval:.2e}' if pval < 0.001 else f'{pval:.4f}'
    dec_color = '#e74c3c' if fail_row else ('#27ae60' if 'reject' in decision else '#555')

    row_vals = [comp, metric, f'{lm:.3f}', f'{rm:.3f}', delta_str, p_str, decision]
    for val, cx in zip(row_vals, col_x):
        color = dec_color if cx == col_x[-1] else '#2c3e50'
        fw = 'bold' if cx == col_x[-1] or fail_row else 'normal'
        ax.text(cx + 0.005, y + 0.0, val, transform=ax.transAxes,
                fontsize=7.2, va='center', color=color, fontweight=fw)

ax.text(0.0, 0.04, '★ = core dissertation claim (Base beats QLoRA at k=3, statistically significant)',
        transform=ax.transAxes, fontsize=8, color='#e67e22', style='italic')
ax.text(0.0, 0.01, '⚠ Llama QLoRA k0→k3 EX uplift p=0.055 — NOT significant (red row)',
        transform=ax.transAxes, fontsize=8, color='#e74c3c', style='italic')

ax.set_title('Paired t-test Results  (α = 0.05, matched by seed + example_id)',
             fontsize=10, fontweight='bold', color='#2c3e50')
plt.tight_layout()
plt.show()

---
## 9. ReAct Results (Preliminary — Single Seed)

The ReAct loop (Llama + QLoRA adapter, k=3, seed=7, n=200) achieves:

| Metric | Baseline k=3 | ReAct | Δ |
|--------|-------------|-------|---|
| VA     | 0.870       | **0.915** | +4.5pp |
| EM     | 0.298       | **0.480** | +18.2pp |
| EX     | 0.517       | **0.605** | +8.8pp |
| TS     | 0.559       | **0.639** | +8.0pp |

> ⚠️ **Single-seed result only** — seeds 17 and 27 needed before paired significance tests are valid.

In [ ]:
# ── ReAct vs Baseline k=3 comparison chart ───────────────────────────────────
metrics = ['VA', 'EM', 'EX', 'TS']
baseline_k3 = [0.870, 0.298, 0.517, 0.559]
react_vals  = [0.915, 0.480, 0.605, 0.639]

x = np.arange(len(metrics))
w = 0.32

fig, ax = plt.subplots(figsize=(8, 4.5), facecolor='#f8f9fa')
ax.set_facecolor('#f8f9fa')

b1 = ax.bar(x - w/2, baseline_k3, w, label='Llama Base k=3', color='#2980b9', alpha=0.88)
b2 = ax.bar(x + w/2, react_vals,  w, label='ReAct (Llama+QLoRA, k=3, seed=7)',
            color='#e67e22', alpha=0.88)

for bars in [b1, b2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.008,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8)

# delta labels
for i, (bv, rv) in enumerate(zip(baseline_k3, react_vals)):
    delta = (rv - bv) * 100
    ax.text(i, max(bv, rv) + 0.045, f'+{delta:.1f}pp',
            ha='center', va='bottom', fontsize=7.5,
            color='#27ae60', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score (0–1)', fontsize=9)
ax.legend(fontsize=8.5)
ax.set_title('ReAct vs Baseline k=3 — Llama-3-8B  (preliminary, single seed)',
             fontsize=10, fontweight='bold', color='#2c3e50')
plt.tight_layout()
plt.show()

---
## 10. Key Findings

### Finding 1 — Few-shot prompting beats fine-tuning (statistically significant)

At k=3, the base (no fine-tuning) model outperforms QLoRA on **execution accuracy** for both model families:

- **Llama**: Base 51.7% vs QLoRA 46.5% — Δ = −5.2pp (p = 0.00074) ✓
- **Qwen**: Base 61.0% vs QLoRA 53.0% — Δ = −8.0pp (p = 8.6 × 10⁻⁶) ✓

This suggests that in a *constrained resource setting* (single GPU, small training set), **domain adaptation via QLoRA does not compensate for the loss of few-shot generalisation capacity**.

### Finding 2 — Llama QLoRA shows attenuated ICL benefit

While Llama Base gains +10.7pp EX from k=0→k=3 (p < 10⁻¹¹), **Llama QLoRA only gains +2.5pp and this is NOT statistically significant** (p = 0.055).  
This implies fine-tuning partially suppresses the model's ability to leverage in-context examples.

### Finding 3 — ReAct adds further gains on top of baseline

The execution-guided repair loop (preliminary, single seed) gains +8.8pp EX over the Llama Base k=3 baseline.  
This positions ReAct as a practical extension layer rather than a replacement for prompting.

---

### Research Journey Summary

```
Zero-shot baseline  →  Few-shot (k=3)  →  QLoRA fine-tuning  →  ReAct loop
    EX ≈ 41–48%        EX ≈ 52–61%        EX ≈ 47–53%          EX ≈ 61%*

        ↑ prompting wins over fine-tuning ↑          ↑ repair adds more ↑
                   (* preliminary, Llama only)
```